In [37]:
import pandas as pd
import joblib 
import numpy as np
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [38]:
# load dataset
dataset = pd.read_csv("../../data/processed/California_Housing.csv")

In [39]:
# Normalize

target = "median_income"
x = dataset.drop(columns=target)
x["ocean_proximity"] = pd.factorize(x["ocean_proximity"])[0]
x["total_bedrooms"] = x["total_bedrooms"].fillna(x["total_bedrooms"].median())
y = dataset[target]

scaler = StandardScaler()

x_train ,x_test , y_train , y_test = train_test_split(x,y,test_size=0.25 , random_state=42)

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

X_train_tensor = torch.tensor(x_train , dtype=torch.float32)
Y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

X_test_tensor = torch.tensor(x_test , dtype=torch.float32)
Y_test_tensor  = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [40]:
#traing process
model = nn.Sequential(
    nn.Linear(X_train_tensor.shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters() , lr = 0.001)

for epoch in range(500):
    y_pred = model(X_train_tensor)
    
    loss = criterion(y_pred, Y_train_tensor)  # Y_train_tensor = float32
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 17.753768920898438
Epoch 20, Loss: 15.20967960357666
Epoch 40, Loss: 8.706018447875977
Epoch 60, Loss: 4.370783805847168
Epoch 80, Loss: 2.842324733734131
Epoch 100, Loss: 2.273503065109253
Epoch 120, Loss: 2.0336050987243652
Epoch 140, Loss: 1.8797270059585571
Epoch 160, Loss: 1.7484827041625977
Epoch 180, Loss: 1.628994107246399
Epoch 200, Loss: 1.517259955406189
Epoch 220, Loss: 1.410753846168518
Epoch 240, Loss: 1.3106955289840698
Epoch 260, Loss: 1.216599702835083
Epoch 280, Loss: 1.1302509307861328
Epoch 300, Loss: 1.0556237697601318
Epoch 320, Loss: 0.9832376837730408
Epoch 340, Loss: 0.9293082356452942
Epoch 360, Loss: 0.8930314779281616
Epoch 380, Loss: 0.8663599491119385
Epoch 400, Loss: 0.8447644114494324
Epoch 420, Loss: 0.8271400928497314
Epoch 440, Loss: 0.8122376799583435
Epoch 460, Loss: 0.7990090250968933
Epoch 480, Loss: 0.7867172956466675


In [41]:
#Evaluation
with torch.no_grad():
    y_test_pred = model(X_test_tensor)
    mse = criterion(y_test_pred, Y_test_tensor)
    rmse = torch.sqrt(mse)
    print("Test RMSE:", rmse.item())
    
print(torch.isnan(Y_train_tensor).sum())  # ถ้า >0 → มี NaN
print(torch.isinf(Y_train_tensor).sum())  # ถ้า >0 → มี Inf
print(torch.isnan(X_train_tensor).sum())  # input ก็ต้อง check

Test RMSE: 1.032942533493042
tensor(0)
tensor(0)
tensor(0)
